# 🧠 ニューラルネットワーク完全ガイド
**パーセプトロンから Transformer まで**

---

## 📚 学習パス
- **Level 1** (初級): パーセプトロン・活性化関数・順伝播
- **Level 2** (中級): 誤差逆伝播法・最適化・正則化
- **Level 3** (上級): CNN / RNN / LSTM / Transformer

## Level 1: 基礎

### 1.1 ニューロンの計算

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

# ── 基本ニューロン ──
def neuron(inputs, weights, bias, activation="sigmoid"):
    z = np.dot(inputs, weights) + bias
    if activation == "sigmoid":
        return 1 / (1 + np.exp(-z))
    elif activation == "relu":
        return max(0, z)
    return z  # linear

# 例: 入力3本
inputs  = np.array([0.5, -0.3, 0.8])
weights = np.array([2.0, -1.5,  0.4])
bias    = 0.1

out = neuron(inputs, weights, bias)
print(f"入力 : {inputs}")
print(f"重み : {weights}")
print(f"バイアス: {bias}")
print(f"出力 (sigmoid): {out:.4f}")

In [ ]:
# ── 活性化関数の比較 ──
x = np.linspace(-5, 5, 200)

activations = {
    "Sigmoid":  1 / (1 + np.exp(-x)),
    "ReLU":     np.maximum(0, x),
    "Tanh":     np.tanh(x),
    "GELU":     x * 0.5 * (1 + np.tanh(np.sqrt(2/np.pi) * (x + 0.044715*x**3))),
}

fig, axes = plt.subplots(1, 4, figsize=(13, 3))
for ax, (name, y) in zip(axes, activations.items()):
    ax.plot(x, y, color="#4dabf7", lw=2)
    ax.axhline(0, color="gray", lw=0.5)
    ax.axvline(0, color="gray", lw=0.5)
    ax.set_title(name); ax.set_ylim(-1.5, 1.5)
plt.suptitle("活性化関数の比較", y=1.02)
plt.tight_layout(); plt.show()

### 1.2 多層ニューラルネットワーク（順伝播）

In [ ]:
# PyTorch で多層 MLP を構築
class MLP(nn.Module):
    def __init__(self, input_dim, hidden_dims, output_dim):
        super().__init__()
        dims = [input_dim] + hidden_dims + [output_dim]
        layers = []
        for i in range(len(dims)-1):
            layers.append(nn.Linear(dims[i], dims[i+1]))
            if i < len(dims)-2:
                layers.append(nn.ReLU())
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

model = MLP(input_dim=3, hidden_dims=[8, 4], output_dim=1)
print(model)
sample = torch.randn(5, 3)
out = model(sample)
print(f"\n入力 shape: {sample.shape} → 出力 shape: {out.shape}")

## Level 2: コア理論

### 2.1 誤差逆伝播法（バックプロパゲーション）

In [ ]:
# XOR を MLP で学習し、損失曲線を可視化
torch.manual_seed(42)

X = torch.tensor([[0,0],[0,1],[1,0],[1,1]], dtype=torch.float32)
y = torch.tensor([[0],[1],[1],[0]], dtype=torch.float32)

model = nn.Sequential(
    nn.Linear(2, 8), nn.ReLU(),
    nn.Linear(8, 4), nn.ReLU(),
    nn.Linear(4, 1), nn.Sigmoid()
)
optim  = torch.optim.Adam(model.parameters(), lr=0.05)
loss_fn = nn.BCELoss()

losses = []
for epoch in range(500):
    pred = model(X)
    loss = loss_fn(pred, y)
    optim.zero_grad(); loss.backward(); optim.step()
    losses.append(loss.item())

plt.figure(figsize=(7, 3))
plt.plot(losses, color="#74c0fc"); plt.xlabel("Epoch"); plt.ylabel("BCE Loss")
plt.title("XOR 問題の学習曲線（バックプロパゲーション）"); plt.tight_layout(); plt.show()

print("\n最終予測:")
with torch.no_grad():
    for xi, yi in zip(X, y):
        p = model(xi.unsqueeze(0)).item()
        print(f"  {xi.tolist()} → {p:.3f}  (正解:{int(yi.item())})")

### 2.2 最適化アルゴリズムの比較

In [ ]:
# SGD / Adam / RMSprop を同じ問題で比較
def run_optimizer(name, optimizer_cls, **kw):
    torch.manual_seed(0)
    m = nn.Sequential(nn.Linear(2,8), nn.ReLU(), nn.Linear(8,1), nn.Sigmoid())
    opt = optimizer_cls(m.parameters(), **kw)
    fn  = nn.BCELoss()
    ls  = []
    for _ in range(300):
        p = m(X); l = fn(p, y)
        opt.zero_grad(); l.backward(); opt.step()
        ls.append(l.item())
    return ls

results = {
    "SGD (lr=0.1)":         run_optimizer("SGD",     torch.optim.SGD,     lr=0.1),
    "SGD+momentum (0.9)":   run_optimizer("SGDM",    torch.optim.SGD,     lr=0.05, momentum=0.9),
    "Adam (lr=0.01)":       run_optimizer("Adam",    torch.optim.Adam,    lr=0.01),
    "RMSprop (lr=0.01)":    run_optimizer("RMSprop", torch.optim.RMSprop, lr=0.01),
}

plt.figure(figsize=(9, 4))
for name, ls in results.items():
    plt.plot(ls, label=name)
plt.xlabel("Epoch"); plt.ylabel("Loss")
plt.title("最適化アルゴリズム比較 (XOR 問題)")
plt.legend(); plt.tight_layout(); plt.show()

## Level 3: 現代アーキテクチャ

### 3.1 Transformer Self-Attention の実装

In [ ]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns

def scaled_dot_product_attention(Q, K, V, mask=None):
    d_k = Q.size(-1)
    scores = torch.matmul(Q, K.transpose(-2, -1)) / d_k**0.5
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float("-inf"))
    attn = F.softmax(scores, dim=-1)
    return torch.matmul(attn, V), attn

torch.manual_seed(1)
seq_len, d_k, d_v = 5, 8, 8

Q = torch.randn(seq_len, d_k)
K = torch.randn(seq_len, d_k)
V = torch.randn(seq_len, d_v)

output, attn_weights = scaled_dot_product_attention(Q, K, V)

tokens = ["The", "cat", "sat", "on", "mat"]
plt.figure(figsize=(6, 5))
sns.heatmap(attn_weights.detach().numpy(), annot=True, fmt=".2f",
            xticklabels=tokens, yticklabels=tokens, cmap="Blues")
plt.title("Self-Attention 重みヒートマップ")
plt.xlabel("Key (注目先)"); plt.ylabel("Query (注目元)")
plt.tight_layout(); plt.show()

### 3.2 このプロジェクトへの応用 - BGE-M3 埋め込み

`corpus/` は FAISS インデックスで BGE-M3 の埋め込みを保存しています。

In [ ]:
# プロジェクトの Retriever を使う（実行にはプロジェクト環境が必要）
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd()))

try:
    from src.rag.retriever import Retriever
    corpus_path = pathlib.Path("corpus")
    retriever = Retriever(
        index_path=str(corpus_path / "corpus.index"),
        meta_path=str(corpus_path / "corpus_meta.json"),
    )
    results = retriever.hybrid_search("ニューラルネットワークの学習", top_k=3)
    print("ハイブリッド検索結果:")
    for r in results:
        print(f"  [{r.get('score', 0):.4f}] {str(r.get('source', ''))[:60]}")
except Exception as e:
    print(f"Retriever ロード不可: {e}")
    print("→ corpus/ にインデックスが存在するか確認してください")

## ✅ 理解度チェック

- [ ] 活性化関数 (Sigmoid / ReLU / GELU) の使い分けが説明できる  
- [ ] バックプロパゲーションが「損失を勾配で逆伝播する」と分かった  
- [ ] Adam が SGD より速く収束することを実験で確認した  
- [ ] Attention ヒートマップの読み方が分かった  

---
✅ 完了 → **[数学基礎ガイド](プロジェクトに必要な数学基礎.ipynb)** へ